# TAD Ethereum — Controller Notebook

**Reimplementation of:** Ofori-Boateng et al. (2021) — *Topological Anomaly Detection in Dynamic Multilayer Blockchain Networks* ([arXiv:2106.01806](https://arxiv.org/abs/2106.01806))

### Pipeline (runs automatically after configuration)
1. Load & filter the transaction graph to top-ranked nodes
2. For each day (and each layer), build a weighted graph → geodesic densification
3. Run clique persistent homology → Persistence Diagram (PD) per day
4. If layers are enabled, stack all layer PDs into a Stacked PD (SPD)
5. Compute Wasserstein distance W₁(PD_{t-1}, PD_t) between consecutive days
6. Extract TDA index features and compute composite indices
7. Save results — one JSON file per year, one key per layer-set

---

**Only Section 1 (Configuration) needs editing between runs.**

## 0. Setup — load function library

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

from tad_ethereum_functions import run_all

print('Setup complete ✓')

## 1. Configuration

**Edit only this cell between runs.**

### `YEARS`
List of years to process. Each year gets its own output file (`run_results_V1_<YEAR>.json`).

### `LAYERS`
Dictionary of named layer-sets. Each key becomes the **run name** (entry key) inside the year's results file.  
Each value is a `LAYER_FILTERS` dict `{layer_name: callable(df) -> bool mask}`, or `None` for single-layer mode.

```python
# Example entries:

'simple_and_contracts': {
    'simple_txs':   lambda d: d['total_input_bytes'] <= 300,
    'contract_txs': lambda d: d['total_input_bytes'] >  300,
},

'by_value': {
    'low_value':  lambda d: d['tx_value'] <= d['tx_value'].median(),
    'high_value': lambda d: d['tx_value'] >  d['tx_value'].median(),
},

'single_layer': None,   # no split — whole graph as one layer
```

### Output layout
```
run_results_V1_2023.json
  └─ "simple_and_contracts"  ← run produced by LAYERS key
  └─ "by_value"

run_results_V1_2024.json
  └─ "simple_and_contracts"
  └─ "by_value"
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  edit everything in this cell
# ════════════════════════════════════════════════════════════════════════════

# ── Years to process ─────────────────────────────────────────────────────────
YEARS = [2020,2021,2022,2023,2024,2025]

# ── Layer-sets ────────────────────────────────────────────────────────────────
# Key   = run name (stored as the key in each year's results JSON)
# Value = LAYER_FILTERS dict, or None for single-layer mode

    # ── Add more layer-sets here, e.g.: ──────────────────────────────────────
    # 'simple_and_contracts': {
    #     'simple_txs':   lambda d: d['total_input_bytes'] <= 300,
    #     'contract_txs': lambda d: d['total_input_bytes'] >  300,
    # },
    # 'single_layer': None,


LAYERS = {
    'contract_txs_ETH_only_750':{'contract_txs_ETH_only': lambda d: (d['tx_value'] == 0) & (d['erc20']=='ETH')},
    'simple_txs_ETH_only_750':{'simple_txs_ETH_only':   lambda d: (d['tx_value'] >  0) & (d['erc20']=='ETH')},
    'contract_txs_ALL_750':{'contract_txs_ALL': lambda d: (d['tx_value'] == 0)},
    'simple_txs_ALL_750':{'simple_txs_ALL':   lambda d: (d['tx_value'] >  0)},
    'contract_factory_ALL_750':{'contract_factory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100)},
    'contract_nonFactory_ALL_750':{'contract_nonFactory_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']<100)},
    'contract_highInput_ALL_750':{'contract_highInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=500)},
    'contract_mediumInput_ALL_750':{'contract_mediumInput_ALL': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['total_input_bytes']<500)}
}



# ── TDA settings ─────────────────────────────────────────────────────────────
TDA_CFG = {
    'edge_weight_col':  'tx_count',        # column used as edge weight
    'max_nodes':        1000,              # node cap per day / layer
    'homology_maxdim':  1,                 # H0 + H1
    'distance_metric':  'wasserstein',     # 'wasserstein' or 'bottleneck'
    'similarity_metric':'norm_similarity', # 'norm_similarity' or 'i/log'
    'alpha':            9,                 # constant in norm_similarity
    'ranking_metric':   'tx_count',        # tx_count / tx_value / page_rank / ...
    'global_top':       0,                 # 0 = skip global filter
    'daily_top':        750,               # keep top-N nodes per day
}

# ── Paths ─────────────────────────────────────────────────────────────────────
# data_root / YEAR / eth_tx_value_output/weekly   ← ETH-native parquets
# data_root / YEAR / erc20_tx_value_output/weekly ← ERC20 parquets
# ranking_root / YEAR / global_top_nodes.parquet
# ranking_root / YEAR / daily/
PATH_CFG = {
    'data_root':      Path('data'),
    'ranking_root':   Path('data/ranking'),
    'results_prefix': 'run_results_V1',          # → run_results_V1_<YEAR>.json
}

print(f'Years      : {YEARS}')
print(f'Layer-sets : {list(LAYERS.keys())}')
print(f'Total runs : {len(YEARS) * len(LAYERS)}')
print(f'Results    : {PATH_CFG["results_prefix"]}_<YEAR>.json')
print('Configuration set ✓')

<!-- --- -->
## 2. Run

Single call — no editing required.

In [ ]:
run_all(
    years    = YEARS,
    layers   = LAYERS,
    tda_cfg  = TDA_CFG,
    path_cfg = PATH_CFG,
)